In [1]:
import nest_asyncio
from rich.progress import track

from sdg.configs.generators import TranslatorGeneratorConfig

nest_asyncio.apply()

In [2]:
import polars as pl

to_process = {
    "drama": [
        "drie015",
        "fran023",
        "lois006",
        "naus001",
        "peen001"
    ],
    "jeugdliteratuur": [
        "goej001",
        "goev001",
        "hoff049",
        "perr041",
        "sche039"
    ],
    "poëzie": [
        "beer008",
        "dool003",
        "goev001",
        "lenn006",
        "sche039"
    ],
    "proza": [
        "_kle007",
        "_kon002",
        "goej001",
        "goev001",
        "pier003"
    ]
}

languages = ["English", "Modern Nederlands"]

DATASET_PATH = f"data/splitted_1850_text_length_max_70k_top_5_most_published"
SAVE_PATH = DATASET_PATH



In [3]:
def process(author, genre, language):
    import os

    dataset_genre_path = os.path.join(DATASET_PATH, genre)
    dataset_author_path = os.path.join(dataset_genre_path, f"{author}.parquet")
    df = pl.read_parquet(dataset_author_path)

    from sdg.configs import ModelConfig
    from sdg.generator import TranslationGenerator

    translator_config = TranslatorGeneratorConfig(
        language=language
    )

    model_config = ModelConfig(
        model_name="gpt-4o-mini",
        temperature=0.1,
        timeout=300,
        max_retries=3
    )

    translator = TranslationGenerator(translator_config, model_config)

    translated_texts = []
    for row in track(df.iter_rows(named=True), total=len(df), description="Translating Chunks...", transient=True):
        text = row["text"]
        translated_text = translator.generate_with_splitting(text, 10000)
        translated_texts.append(translated_text.translated_text)

    # output_translated_texts = await translator.batch(df["text"].to_list())
    #
    # if len(df) != len(output_translated_texts):
    #     raise ValueError(f"Error lenghts dont match. DF: {len(df)}, Translated DF (objects): {len(translated_texts)}")
    #
    # for translated_text in output_translated_texts:
    #     translated_texts.append(translated_text.translated_text)

    if len(df) != len(translated_texts):
        raise ValueError(f"Error lengths dont match. DF: {len(df)}, Translated DF (strings): {len(translated_texts)}")

    df = df.with_columns(pl.Series(name=f"{translator_config.language.lower().replace(' ', '_')}_translation", values=translated_texts))

    save_genre_path = os.path.join(SAVE_PATH, genre)
    save_author_path = os.path.join(save_genre_path, f"{author}.parquet")

    os.makedirs(save_genre_path, exist_ok=True)
    df.write_parquet(save_author_path)

Output()

In [ ]:
failed_count = 0
failed_info = {}

for language in track(languages, total=len(languages), description="Processing Language..."):
    for genre, authors in track(to_process.items(), total=len(to_process.items()), description=f"Translating Genre Books into {language}...", transient=True):
        for author in track(authors, total=len(authors), description=f"Translating {genre} Author Books into {language}...", transient=True):
            try:
                process(author, genre, language)
            except Exception as e:
                failed_count += 1
                if language not in failed_info:
                    failed_info[language] = {}
                if genre not in failed_info[language]:
                    failed_info[language][genre] = {}
                if author not in failed_info[language][genre]:
                    failed_info[language][genre][author] = 0
                failed_info[language][genre][author] += 1